In [2]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

/opt/anaconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

True

In [4]:
llm=ChatGroq(model="llama-3.1-8b-instant")
parser=StrOutputParser()
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])
chain=prompt|llm|parser
K=4
store={}
def get_session_history(session_id:str)->ChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]


In [7]:
def get_window_history(session_id:str)->ChatMessageHistory:
    history=get_session_history(session_id)
    if len(history.messages)>K:
        trimmed=ChatMessageHistory()
        for msg in history.messages[-K:]:
            trimmed.add_message(msg)
        return trimmed
    return history

In [12]:
chain_with_memory=RunnableWithMessageHistory(chain,get_window_history,input_messages_key="input",history_messages_key="history")
def chat(message, session_id="session_1"):
    response = chain_with_memory.invoke(
        {"input": message},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"You: {message}")
    print(f"AI : {response}")

/opt/anaconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3701: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [18]:
session_id = "session_1"
real_history=store.get(session_id,ChatMessageHistory())
total=len(real_history.messages)
window = real_history.messages[-K:] if total > K else real_history.messages
print(f"Memory: showing last {len(window)} of {total} total messages")
print()

Memory: showing last 4 of 6 total messages



In [15]:
print("═" * 50)
print("FILE 2: Buffer Window Memory (K=4 messages = 2 exchanges)")
print("═" * 50)
print()

══════════════════════════════════════════════════
FILE 2: Buffer Window Memory (K=4 messages = 2 exchanges)
══════════════════════════════════════════════════



In [17]:

chat("My name is Krish.")
chat("I live in Ahmedabad.")
chat("I am 21 years old.")          # Turn 3 — Turn 1 starts dropping
chat("I love playing cricket.")     # Turn 4 — Turn 1 fully dropped
chat("What is my name?")            # Turn 5 — can it remember? NO
chat("What city do I live in?")     # Turn 6 — also forgotten

print("═" * 50)
print(f"K={K} — only last {K} messages sent to LLM")
print("Old messages dropped — that is the tradeoff")
print("═" * 50)

You: My name is Krish.
AI : Nice to meet you, Krish. Ahmedabad is a wonderful place to be young and exploring your interests. What do you like to do in your free time? Are you into sports, music, or maybe trying out new foods?
You: I live in Ahmedabad.
AI : So, you live in Ahmedabad and are 22 years old. Are you looking for information about the city, its attractions, or perhaps places to visit or things to do? Or maybe you need help with something specific, like a recommendation for a good restaurant or a way to get around the city?
You: I am 21 years old.
AI : So you're 21 years old, which means you're just a year shy of the big 2-2. As a young adult living in Ahmedabad, you must have plenty of opportunities to explore the city and its surroundings. What are some things you're interested in or enjoy doing in your free time?
You: I love playing cricket.
AI : Cricket is a very popular sport in India, and Ahmedabad is no exception. The city has a rich cricketing history, with the Sardar